# DESIGN REPAIR / LOCAL INPAINT

`RUN 003 -> mascara da roupa -> Waifu-Inpaint-XL -> roupa corrigida`

Linha experimental **separada**. O benchmark do `waiIllustriousSDXL_v170` continua valido e intocado; este checkpoint nao o substitui.

A Run 003 **nao esta no Git** — e enviada por upload na celula 2.

Uma unica execucao, parametros conservadores, sem sweep.


In [ ]:
#@title 0. PAINEL — DESIGN REPAIR / LOCAL INPAINT { display-mode: "form" }
#@markdown # WAIFU-INPAINT-XL — correcao LOCAL de roupa
#@markdown
#@markdown Linha experimental **DESIGN REPAIR / LOCAL INPAINT**, separada do
#@markdown benchmark do `waiIllustriousSDXL_v170`, que permanece intacto.
#@markdown
#@markdown `RUN 003 -> mascara da roupa -> Waifu-Inpaint-XL -> roupa corrigida`
#@markdown
#@markdown Edita SOMENTE a regiao mascarada. Nao gera personagem nova.

#@markdown ---
#@markdown ### PERSONAGEM (parametrizado — serve para outras waifus)
CHARACTER_ID = "waifu_001"  #@param {type:"string"}
#@markdown Nome do arquivo que voce vai subir como imagem de partida.
SOURCE_IMAGE = "run_003_output.png"  #@param {type:"string"}
#@markdown Mascara da roupa, no ESPACO DA RUN 003 (nao da arte original).
MASK_IMAGE = "outfit_mask.png"  #@param {type:"string"}
#@markdown Mascara das regioes protegidas (opcional, mas recomendada):
#@markdown rosto, olhos, cabelo, chifres, maos. Se fornecida, o overlap com
#@markdown a mascara de roupa tem de ser ZERO.
PROTECTED_MASK = "protected_mask.png"  #@param {type:"string"}
#@markdown Referencia contextual. NAO usada no inpaint puro.
FULL_BODY_REFERENCE = "full_body.png"  #@param {type:"string"}

#@markdown ---
#@markdown ### MODO
#@markdown `PURE_INPAINT` e o unico implementado. O checkpoint de inpainting
#@markdown nao tem mecanismo proprio de referencia; usar `full_body.png` como
#@markdown referencia exigiria IP-Adapter, que NAO entra neste teste.
INPAINT_MODE = "PURE_INPAINT"  #@param ["PURE_INPAINT", "WITH_REFERENCE"]

#@markdown ---
#@markdown ### MASCARA
MASK_DILATION = 8  #@param {type:"slider", min:0, max:64, step:1}
MASK_FEATHER = 6  #@param {type:"slider", min:0, max:64, step:1}
MASK_CHANNEL = "red"  #@param ["red", "green", "blue", "alpha"]
MASK_INVERT = False  #@param {type:"boolean"}

#@markdown ---
#@markdown ### PROMPT (fala da funcao do reparo, nunca da personagem)
PROMPT_PRESET = "outfit_repair_v1"  #@param ["outfit_repair_v1"]
USAR_PROMPT_CUSTOM = False  #@param {type:"boolean"}
PROMPT_CUSTOM = "1girl, solo, full body, chibi, super deformed, clean lineart, anime coloring, simple cel shading, detailed outfit, detailed costume, preserve character design"  #@param {type:"string"}
#@markdown
USAR_NEGATIVE_CUSTOM = False  #@param {type:"boolean"}
NEGATIVE_CUSTOM = "bad quality, worst quality, worst detail, sketch, watermark, signature, logo, text, realistic, photorealistic, 3d, nude, naked, different outfit, changed design, extra limbs, deformed"  #@param {type:"string"}

#@markdown ---
#@markdown ### PARAMETROS CONSERVADORES (primeiro teste — sem sweep)
INPAINT_STRENGTH = 0.75  #@param {type:"slider", min:0.10, max:1.00, step:0.05}
STEPS = 28  #@param {type:"slider", min:10, max:60, step:1}
CFG_SCALE = 5.0  #@param {type:"slider", min:1.0, max:12.0, step:0.5}
SAMPLER = "euler_ancestral"  #@param {type:"string"}
SCHEDULER = "normal"  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}

#@markdown ---
#@markdown ### V-PREDICTION (obrigatorio — linhagem WAI V14 V-Pred)
#@markdown Configuracao epsilon produz imagem queimada. Nao usar.
SAMPLING_TYPE = "v_prediction"  #@param ["v_prediction", "eps"]
ZSNR = False  #@param {type:"boolean"}

EXPERIMENT_LABEL = ""  #@param {type:"string"}

# ----------------------------------------------------------------------
import datetime, pathlib

EXPERIMENT_LINE = "DESIGN_REPAIR_LOCAL_INPAINT"
MODEL_KEY = "waifu_inpaint_xl"
CHECKPOINT_FILE = "Waifu-Inpaint-XL.safetensors"
WORKFLOW = "experimental/waifu_inpaint_xl"

# Este checkpoint NAO e o do benchmark. Trocar por WAI v17/v14 comum ou
# outro Illustrious quebraria o experimento: nenhum deles tem UNet de 9
# canais, e o resultado nao seria inpaint de verdade.
CHECKPOINT_SUBSTITUTOS_PROIBIDOS = [
    "waiIllustriousSDXL_v170.safetensors",
    "WAI-NSFW-illustrious-SDXL-V14.0.safetensors",
]
assert CHECKPOINT_FILE not in CHECKPOINT_SUBSTITUTOS_PROIBIDOS

assert SAMPLING_TYPE == "v_prediction", (
    "BLOCKED — este checkpoint tem linhagem V-Prediction. Com eps o output "
    "sai queimado. Nao usar configuracao epsilon.")

assert INPAINT_MODE == "PURE_INPAINT", (
    f"BLOCKED — INPAINT_MODE={INPAINT_MODE} exigiria IP-Adapter, que NAO "
    "entra neste teste. O checkpoint de inpainting nao consome full_body.png "
    "como referencia por conta propria; prometer isso e rodar inpaint puro "
    "seria registrar um experimento que nao aconteceu.")

assert 0.0 < INPAINT_STRENGTH <= 1.0

PROMPT_PRESETS = {
    "outfit_repair_v1": {
        "positive": (
            "1girl, solo, full body, chibi, super deformed, clean lineart, "
            "anime coloring, simple cel shading, detailed outfit, "
            "detailed costume, preserve character design"),
        "negative": (
            "bad quality, worst quality, worst detail, sketch, watermark, "
            "signature, logo, text, realistic, photorealistic, 3d, nude, "
            "naked, different outfit, changed design, extra limbs, deformed"),
    },
}
_preset = PROMPT_PRESETS[PROMPT_PRESET]

def _escolhe(usar, texto, padrao, lado):
    if not usar:
        return padrao, f"preset:{PROMPT_PRESET}"
    t = texto.strip()
    assert t, f"USAR_{lado}_CUSTOM marcado mas o campo esta vazio."
    return (padrao, f"preset:{PROMPT_PRESET}") if t == padrao.strip() else (t, "manual_override")

PROMPT, _sp = _escolhe(USAR_PROMPT_CUSTOM, PROMPT_CUSTOM, _preset["positive"], "PROMPT")
NEGATIVE, _sn = _escolhe(USAR_NEGATIVE_CUSTOM, NEGATIVE_CUSTOM, _preset["negative"], "NEGATIVE")
PROMPT_SOURCE = {"positive": _sp, "negative": _sn}
PROMPT_EDITADO = "manual_override" in PROMPT_SOURCE.values()

UPLOAD_DIR = pathlib.Path("/content/inpaint_inputs")
EVAL_ROOT = pathlib.Path("/content/ChibiCreate/experiments/design_repair")
STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_NAME = EXPERIMENT_LABEL.strip() or f"repair_s{INPAINT_STRENGTH:.2f}"
EXPERIMENT_DIR_NAME = f"eval_{STAMP}"

CONFIG = {
    "experiment_line": EXPERIMENT_LINE,
    "experiment": EXPERIMENT_NAME,
    "experiment_dir": EXPERIMENT_DIR_NAME,
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "character_id": CHARACTER_ID,
    "model_key": MODEL_KEY,
    "checkpoint_file": CHECKPOINT_FILE,
    "not_a_substitute_for": "wai_illustrious_sdxl_v170",
    "separation_note": (
        "Linha experimental propria. O benchmark do WAI v17 continua valido "
        "e intocado; este checkpoint nao o substitui."),
    "pipeline": "sdxl_inpaint_9ch",
    "inpaint_mode": INPAINT_MODE,
    "source_image": SOURCE_IMAGE,
    "source_role": "run_003_output_uploaded",
    "mask_image": MASK_IMAGE,
    "mask_space": "run_003",
    "mask_space_note": (
        "A mascara esta no espaco da RUN 003, nao no da arte original: o "
        "alvo da edicao ja e a run 003."),
    "protected_mask": PROTECTED_MASK,
    "reference_declared_not_used": FULL_BODY_REFERENCE,
    "prompt": PROMPT, "negative_prompt": NEGATIVE,
    "prompt_source": PROMPT_SOURCE,
    "prompt_manually_edited": PROMPT_EDITADO,
    "prompt_type": "generic_outfit_repair",
    "character_specific_prompt": False,
    "mask_params": {"dilation": int(MASK_DILATION), "feather": int(MASK_FEATHER),
                    "channel": MASK_CHANNEL},
    "sampling": {"seed": int(SEED), "steps": int(STEPS), "cfg": float(CFG_SCALE),
                 "sampler": SAMPLER, "scheduler": SCHEDULER,
                 "sampling_type": SAMPLING_TYPE, "zsnr": bool(ZSNR)},
    "inpaint_strength": float(INPAINT_STRENGTH),
    "status": "EXPERIMENTAL_FIRST_RUN",
}

print("=" * 66)
print("DESIGN REPAIR / LOCAL INPAINT —", EXPERIMENT_NAME)
print("=" * 66)
print("Linha separada. O benchmark WAI v17 NAO e alterado nem substituido.")
print()
print("checkpoint :", CHECKPOINT_FILE, "| 9 canais |", SAMPLING_TYPE,
      "| zsnr", ZSNR)
print("source     :", SOURCE_IMAGE, "(upload; nao vem do Git)")
print("mask       :", MASK_IMAGE, "-> espaco da RUN 003")
print("protegida  :", PROTECTED_MASK, "-> overlap exigido = 0")
print("referencia :", FULL_BODY_REFERENCE, "-> DECLARADA, nao usada")
print()
print("PROMPT   :", PROMPT[:90] + ("..." if len(PROMPT) > 90 else ""))
print("  origem :", PROMPT_SOURCE["positive"])
print("NEGATIVO :", NEGATIVE[:90] + ("..." if len(NEGATIVE) > 90 else ""))
print("  origem :", PROMPT_SOURCE["negative"])
if PROMPT_EDITADO:
    print("  [prompt editado a mao — sera validado na celula 5]")
print()
print("modo       :", INPAINT_MODE, "(sem IP-Adapter)")
print("mascara    : dilation", MASK_DILATION, "| feather", MASK_FEATHER)
print("sampling   : steps", STEPS, "| cfg", CFG_SCALE, "|", SAMPLER,
      "| seed", SEED, "| strength", INPAINT_STRENGTH)
print()
print("Uma unica execucao. Sem sweep.")


In [ ]:
#@title 1. Repositorio e ambiente { display-mode: "form" }
#@markdown O codigo desta fase vive na branch de trabalho, nao na `main`.
REPO_BRANCH = "arena/01a07ece-chibicreate"  #@param {type:"string"}
import subprocess, pathlib, sys
try:
    _gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                           "--format=csv,noheader"], capture_output=True,
                          text=True).stdout.strip()
except FileNotFoundError:
    _gpu = ""
print(_gpu or "GPU NAO DETECTADA — Ambiente de execucao > Alterar > GPU")
REPO = pathlib.Path("/content/ChibiCreate")
URL = "https://github.com/BloomRX/ChibiCreate.git"
if REPO.exists():
    subprocess.run(["git","-C",str(REPO),"fetch","-q","origin",REPO_BRANCH], check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","-q","-B",REPO_BRANCH,
                    "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git","clone","-q","--branch",REPO_BRANCH,URL,str(REPO)],
                   check=True)

ramo = subprocess.run(["git","-C",str(REPO),"rev-parse","--abbrev-ref","HEAD"],
                      capture_output=True, text=True).stdout.strip()
commit = subprocess.run(["git","-C",str(REPO),"rev-parse","HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("branch:", ramo)
print("commit:", commit)

SCRIPTS = REPO / "scripts"
if not (SCRIPTS / "chibi" / "inpaint_check.py").exists():
    raise SystemExit(
        f"BLOCKED — {SCRIPTS}/chibi nao existe neste clone.\n"
        f"A branch '{REPO_BRANCH}' foi baixada? A `main` do repositorio so tem "
        "o README, entao clonar sem --branch deixa o notebook sem codigo.")
sys.path.insert(0, str(SCRIPTS))
import chibi.inpaint_check as _probe  # falha aqui e falha cedo, com contexto
print("scripts/chibi importavel:", pathlib.Path(_probe.__file__).name)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("suba os arquivos na proxima celula ->", UPLOAD_DIR)


In [ ]:
#@title 2. Upload da SOURCE, da MASK e (opcional) da PROTECTED { display-mode: "form" }
#@markdown A Run 003 **nao esta no Git** — precisa ser enviada aqui.
#@markdown Envie de uma vez: `run_003_output.png`, `outfit_mask.png` e,
#@markdown se tiver, `protected_mask.png`.
import shutil, pathlib
try:
    from google.colab import files
    subidos = files.upload()
    for nome, dados in subidos.items():
        (UPLOAD_DIR / nome).write_bytes(dados)
        print("recebido:", nome, len(dados), "bytes")
except Exception as e:
    print("[fora do Colab] copie os arquivos manualmente para", UPLOAD_DIR, "|", e)

print()
print("conteudo de", UPLOAD_DIR)
for p in sorted(UPLOAD_DIR.iterdir()):
    print("  ", p.name, p.stat().st_size, "bytes")


In [ ]:
#@title 3. Validar SOURCE e MASK (para se algo estiver errado) { display-mode: "form" }
#@markdown Mostra SOURCE, MASK e OVERLAY. Bloqueia se a mascara nao for
#@markdown valida — e a mascara e o fator mais critico do experimento.
import hashlib, json, sys, numpy as np
from PIL import Image
sys.path.insert(0, str(SCRIPTS))
from chibi.inpaint_check import overlay as _overlay, overlap_protegido
import matplotlib.pyplot as plt

SRC = UPLOAD_DIR / SOURCE_IMAGE
MSK = UPLOAD_DIR / MASK_IMAGE
PROT = UPLOAD_DIR / PROTECTED_MASK

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

faltando = [p.name for p in (SRC, MSK) if not p.exists()]
if faltando:
    raise SystemExit(
        f"BLOCKED — nao encontrei {faltando} em {UPLOAD_DIR}.\n"
        "A Run 003 nao esta no Git: use a celula 2 para subir os arquivos.")

src_img = Image.open(SRC).convert("RGB")
_msk_raw = Image.open(MSK)


def _extrair_canal(img, canal):
    """Le a mascara do canal escolhido no painel.

    Um PNG com transparencia exportado de editor de camadas costuma ter RGB
    branco em TODA a tela e a forma pintada apenas no alpha. Converter esse
    arquivo para "L" ignora o alpha e devolve uma mascara cheia.
    """
    if canal == "alpha":
        if "A" not in img.getbands():
            raise SystemExit(
                f"BLOCKED — MASK_CHANNEL='alpha' mas {MSK.name} nao tem canal "
                f"alpha (bandas: {''.join(img.getbands())}). "
                "Use 'red' ou reexporte preservando a transparencia.")
        return img.getchannel("A")
    if canal in ("red", "green", "blue"):
        if img.mode in ("L", "1", "I;16"):
            return img.convert("L")
        return img.convert("RGB").getchannel({"red": "R", "green": "G",
                                              "blue": "B"}[canal])
    raise SystemExit(f"BLOCKED — MASK_CHANNEL desconhecido: {canal!r}")


def _perfil(img):
    """% de area branca que cada canal produziria — para diagnostico."""
    linhas = []
    for banda in img.getbands():
        arr = np.asarray(img.getchannel(banda))
        linhas.append(f"    {banda}: {100.0 * (arr > 127).mean():6.2f}% claro")
    return "\n".join(linhas)


msk_img = _extrair_canal(_msk_raw, MASK_CHANNEL)
if MASK_INVERT:
    msk_img = Image.eval(msk_img, lambda v: 255 - v)
print("MASK bandas:", "".join(_msk_raw.getbands()),
      "| canal lido:", MASK_CHANNEL, "| invertida:", MASK_INVERT)

SOURCE_SHA256 = _sha(SRC)
SOURCE_PIXEL_SHA256 = hashlib.sha256(
    np.asarray(src_img).tobytes()).hexdigest()
MASK_SHA256 = _sha(MSK)

print("SOURCE :", SRC.name, src_img.size)
print("  artifact sha256:", SOURCE_SHA256)
print("  pixel    sha256:", SOURCE_PIXEL_SHA256)
print("MASK   :", MSK.name, msk_img.size)
print("  sha256:", MASK_SHA256)

if msk_img.size != src_img.size:
    raise SystemExit(
        f"BLOCKED — mask {msk_img.size} != source {src_img.size}. "
        "Redimensionar aqui deslocaria a regiao editavel. A mascara tem de "
        "ser desenhada NO ESPACO DA RUN 003.")

m = np.asarray(msk_img) > 127
MASK_AREA_PCT = round(100.0 * m.sum() / m.size, 3)
print("area editavel:", MASK_AREA_PCT, "%")
if not m.any():
    raise SystemExit("BLOCKED — mascara vazia: nada a editar.")
if MASK_AREA_PCT > 60:
    raise SystemExit(
        f"BLOCKED — {MASK_AREA_PCT}% da imagem marcado como editavel. "
        "Deixa de ser correcao local e vira reinterpretacao da personagem.\n"
        "\nSe voce pintou so uma parte pequena, o arquivo provavelmente esta "
        "sendo lido pelo canal errado. Perfil dos canais de "
        f"{MSK.name}:\n{_perfil(_msk_raw)}\n"
        "\nO canal com a porcentagem parecida com o que voce pintou e o "
        "certo. Ajuste MASK_CHANNEL na celula 0 (use 'alpha' se voce pintou "
        "numa camada transparente) ou marque MASK_INVERT se voce pintou de "
        "preto sobre fundo branco. Alternativa: reexporte a mascara achatada, "
        "branco na roupa e preto no resto, sem transparencia.")
if MASK_AREA_PCT < 1:
    raise SystemExit(
        f"BLOCKED — so {MASK_AREA_PCT}% marcado como editavel. Provavel erro "
        f"de canal ou de exportacao. Perfil dos canais de "
        f"{MSK.name}:\n{_perfil(_msk_raw)}\n"
        "Ajuste MASK_CHANNEL ou MASK_INVERT na celula 0.")

PROTECTED_OVERLAP_PIXELS = None
if PROT.exists():
    prot_img = Image.open(PROT).convert("L")
    if prot_img.size != src_img.size:
        raise SystemExit(f"BLOCKED — protected {prot_img.size} != source {src_img.size}")
    PROTECTED_MASK_SHA256 = _sha(PROT)
    PROTECTED_OVERLAP_PIXELS = overlap_protegido(msk_img, prot_img)
    print("overlap com regiao protegida:", PROTECTED_OVERLAP_PIXELS, "px")
    if PROTECTED_OVERLAP_PIXELS != 0:
        raise SystemExit(
            f"BLOCKED — a mascara invade {PROTECTED_OVERLAP_PIXELS} pixels de "
            "regiao protegida (rosto, olhos, cabelo, chifres, maos). "
            "Exigencia: ZERO. Corrija a mascara e suba de novo.")
else:
    PROTECTED_MASK_SHA256 = None
    print("[aviso] protected_mask.png nao enviada: o overlap com regioes")
    print("        protegidas NAO pode ser verificado automaticamente.")
    print("        A conferencia visual abaixo passa a ser a unica garantia.")

OVERLAY = _overlay(src_img, msk_img)
fig, ax = plt.subplots(1, 3, figsize=(15, 5.5))
for a, img, t in ((ax[0], src_img, "SOURCE (run 003)"),
                  (ax[1], msk_img, "MASK"),
                  (ax[2], OVERLAY, "OVERLAY (vermelho = editavel)")):
    a.imshow(img, cmap="gray" if img.mode == "L" else None)
    a.set_title(t); a.axis("off")
fig.tight_layout(); plt.show()

print()
print("[HUMAN REVIEW REQUIRED] o vermelho cobre SO a roupa?")
print("  Deve ficar de fora: rosto, olhos, cabelo, chifres, maos, pele e")
print("  membros fora da roupa. Se invadir, pare e corrija a mascara.")


In [ ]:
#@title 4. ComfyUI + checkpoint (download via HF_TOKEN) { display-mode: "form" }
#@markdown Sobe o ComfyUI e garante o `Waifu-Inpaint-XL.safetensors`.
#@markdown
#@markdown O repositorio e **gated**: exige que VOCE aceite as condicoes na
#@markdown pagina do modelo, logado na sua conta. Feito o aceite, o download
#@markdown pode ser automatico usando um token seu.
#@markdown
#@markdown **NAO cole o token aqui.** Painel lateral > chave 🔑 (Secrets) >
#@markdown `+ Adicionar novo secret`, nome **HF_TOKEN**, e ligue o acesso a
#@markdown este notebook. Um token so de leitura basta.
BAIXAR_SE_FALTAR = True  #@param {type:"boolean"}

import subprocess, pathlib, time, urllib.request, urllib.error, json, hashlib, os, shutil

COMFY = pathlib.Path("/content/ComfyUI")
if not COMFY.exists():
    subprocess.run(["git","clone","-q","https://github.com/comfyanonymous/ComfyUI.git",
                    str(COMFY)], check=True)
    subprocess.run(["pip","install","-q","-r",str(COMFY/"requirements.txt")], check=True)
COMFY_COMMIT = subprocess.run(["git","-C",str(COMFY),"rev-parse","HEAD"],
                              capture_output=True, text=True).stdout.strip()

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print("[Drive]", e)

CKPT_DIR = pathlib.Path("/content/drive/MyDrive/ComfyUI_Data/models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT = CKPT_DIR / CHECKPOINT_FILE
HF_REPO = "ShinoharaHare/Waifu-Inpaint-XL"
HF_URL = f"https://huggingface.co/{HF_REPO}/resolve/main/{CHECKPOINT_FILE}"


def _hf_token():
    """Colab Secrets -> variavel de ambiente. Nunca escrito no notebook."""
    try:
        from google.colab import userdata
        t = (userdata.get("HF_TOKEN") or "").strip()
        if t:
            return t, "Colab Secrets"
    except Exception:
        pass
    t = os.environ.get("HF_TOKEN", "").strip()
    return (t, "variavel de ambiente") if t else ("", "")


def pedir_token():
    """Alternativa: digitar o token na hora, sem deixar rastro no notebook."""
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("Cole o token (nao aparece): ").strip()
    print("token definido para esta sessao. Rode esta celula de novo.")


def _baixar(url, destino, token):
    """Download com barra simples. Nao deixa arquivo parcial no lugar final."""
    parcial = destino.with_suffix(destino.suffix + ".part")
    req = urllib.request.Request(
        url, headers={"Authorization": f"Bearer {token}",
                      "User-Agent": "ChibiCreate/inpaint-eval"})
    with urllib.request.urlopen(req, timeout=120) as r, open(parcial, "wb") as f:
        total = int(r.headers.get("Content-Length") or 0)
        baixado, marca = 0, time.time()
        while True:
            bloco = r.read(1 << 22)
            if not bloco:
                break
            f.write(bloco)
            baixado += len(bloco)
            if time.time() - marca > 5:
                pct = f"{100*baixado/total:.1f}%" if total else "?"
                print(f"    {baixado/1e9:.2f} GB / {total/1e9:.2f} GB ({pct})")
                marca = time.time()
    obtido = parcial.stat().st_size
    if total and obtido != total:
        parcial.unlink(missing_ok=True)
        raise SystemExit(
            f"BLOCKED — download incompleto: {obtido} de {total} bytes. "
            "Arquivo parcial descartado para nao virar um checkpoint corrompido. "
            "Rode a celula de novo para tentar outra vez.")
    parcial.rename(destino)


if not CKPT.exists() and BAIXAR_SE_FALTAR:
    tok, origem = _hf_token()
    if not tok:
        raise SystemExit(
            f"BLOCKED — {CHECKPOINT_FILE} nao esta no Drive e nao ha HF_TOKEN.\n\n"
            "O repositorio e gated. Dois passos, os dois na SUA conta:\n"
            f"  1. abra https://huggingface.co/{HF_REPO} e aceite as condicoes\n"
            "     (o aceite e pessoal; nao pode ser feito pelo agente)\n"
            "  2. crie um token de leitura em huggingface.co/settings/tokens\n"
            "     e guarde no Colab: painel lateral > chave (Secrets) >\n"
            "     nome HF_TOKEN, com acesso a este notebook\n\n"
            "Alternativa sem secret: rode `pedir_token()` numa celula nova e\n"
            "repita esta. Ou baixe manualmente para\n"
            f"  {CKPT_DIR}")

    print(f"HF_TOKEN carregado de: {origem} (...{tok[-4:]})")
    livre = shutil.disk_usage(CKPT_DIR).free
    print(f"espaco livre no Drive: {livre/1e9:.1f} GB")
    if livre < 8e9:
        raise SystemExit(
            f"BLOCKED — {livre/1e9:.1f} GB livres; o arquivo tem ~6.94 GB. "
            "Libere espaco antes: um download truncado gera checkpoint invalido.")

    print(f"baixando {CHECKPOINT_FILE} de {HF_REPO} ...")
    try:
        _baixar(HF_URL, CKPT, tok)
    except urllib.error.HTTPError as e:
        if e.code in (401, 403):
            raise SystemExit(
                f"BLOCKED — HTTP {e.code} do HuggingFace.\n\n"
                "O token existe, mas nao tem acesso a este modelo. Quase sempre\n"
                "significa que as CONDICOES ainda nao foram aceitas:\n"
                f"  https://huggingface.co/{HF_REPO}\n"
                "Aceite logado na mesma conta do token e rode de novo.\n"
                "Confira tambem se o token tem permissao de leitura.")
        raise
    print("download concluido.")

if not CKPT.exists():
    raise SystemExit(
        f"BLOCKED — {CHECKPOINT_FILE} nao encontrado em {CKPT_DIR}.\n"
        "Marque BAIXAR_SE_FALTAR ou coloque o arquivo manualmente.\n\n"
        "NAO substitua por WAI v17, WAI v14 comum ou outro Illustrious:\n"
        "nenhum tem UNet de 9 canais e o resultado nao seria inpaint real.")

_h = hashlib.sha256()
with open(CKPT, "rb") as f:
    for b in iter(lambda: f.read(1 << 22), b""):
        _h.update(b)
CHECKPOINT_SHA256 = _h.hexdigest()
CHECKPOINT_BYTES = CKPT.stat().st_size
print("checkpoint:", CKPT.name, CHECKPOINT_BYTES, "bytes")
print("sha256    :", CHECKPOINT_SHA256)
print()
print("[registrar em config/models.lock.yaml: weights.sha256 e o tamanho real]")

# safetensors comeca com um cabecalho JSON de 8 bytes little-endian. Um HTML
# de erro salvo com nome .safetensors passaria despercebido ate o ComfyUI
# falhar com erro obscuro.
with open(CKPT, "rb") as f:
    _n = int.from_bytes(f.read(8), "little")
    _ok = 0 < _n < CHECKPOINT_BYTES and f.read(1) == b"{"
if not _ok:
    raise SystemExit(
        "BLOCKED — o arquivo nao parece um safetensors valido. Provavel "
        "download interrompido ou pagina de erro salva com nome de modelo. "
        "Apague e baixe de novo.")

os.makedirs(COMFY/"models"/"checkpoints", exist_ok=True)
alvo = COMFY/"models"/"checkpoints"/CHECKPOINT_FILE
if not alvo.exists():
    os.symlink(CKPT, alvo)

subprocess.run("pkill -f 'main.py --listen' || true", shell=True)
time.sleep(3)
LOG = open("/content/comfy.log", "w")
subprocess.Popen(["python","main.py","--listen","127.0.0.1","--port","8188"],
                 cwd=str(COMFY), stdout=LOG, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/object_info", timeout=5)
        print("ComfyUI no ar | commit", COMFY_COMMIT[:10]); break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("BLOCKED — ComfyUI nao subiu. Ver /content/comfy.log")


In [ ]:
#@title 5. Validar nodes e o grafo { display-mode: "form" }
import json, urllib.request, pathlib, hashlib

OBJECT_INFO = json.load(urllib.request.urlopen(
    "http://127.0.0.1:8188/object_info", timeout=120))

falhas = []
def checa(cond, ok, erro):
    print(("  OK    " if cond else "  FALHA ") + (ok if cond else erro))
    if not cond: falhas.append(erro)

print("NODES")
for n in ["CheckpointLoaderSimple","ModelSamplingDiscrete","CLIPTextEncode",
          "LoadImage","LoadImageMask","GrowMask","FeatherMask",
          "InpaintModelConditioning","KSampler","VAEDecode","SaveImage"]:
    checa(n in OBJECT_INFO, f"{n} disponivel", f"{n} AUSENTE")

def _opcoes(node, campo):
    for g in ("required","optional"):
        spec = OBJECT_INFO.get(node,{}).get("input",{}).get(g,{})
        if campo in spec and isinstance(spec[campo][0], list):
            return spec[campo][0]
    return None

print()
print("PARAMETROS (contra o servidor real)")
for node, campo, valor in (("KSampler","sampler_name",SAMPLER),
                           ("KSampler","scheduler",SCHEDULER),
                           ("ModelSamplingDiscrete","sampling",SAMPLING_TYPE),
                           ("LoadImageMask","channel",MASK_CHANNEL)):
    opts = _opcoes(node, campo)
    if opts is None:
        checa(False,"",f"{node}.{campo} ausente no /object_info")
    else:
        checa(valor in opts, f"{node}.{campo}={valor}",
              f"{node}.{campo}={valor} inexistente. Disponiveis: {opts}")

checa(CHECKPOINT_FILE in (_opcoes("CheckpointLoaderSimple","ckpt_name") or []),
      f"{CHECKPOINT_FILE} visivel para o loader",
      f"{CHECKPOINT_FILE} nao aparece no CheckpointLoaderSimple")

WF_PATH = pathlib.Path(f"/content/ChibiCreate/workflows/{WORKFLOW}/v0.json")
WORKFLOW_SHA256 = hashlib.sha256(WF_PATH.read_bytes()).hexdigest()
WF = {k:v for k,v in json.loads(WF_PATH.read_text()).items() if not k.startswith("_")}
cls = {k:v["class_type"] for k,v in WF.items()}

print()
print("GRAFO")
ks = [k for k,c in cls.items() if c=="KSampler"][0]
imc = [k for k,c in cls.items() if c=="InpaintModelConditioning"]
checa(len(imc)==1, "InpaintModelConditioning presente", "sem InpaintModelConditioning")
checa("VAEEncodeForInpaint" not in cls.values(), "nao usa VAEEncodeForInpaint",
      "VAEEncodeForInpaint exigiria denoise 1.0 e destruiria o design")
checa(WF[ks]["inputs"]["latent_image"][0] in imc,
      "latente vem do InpaintModelConditioning", "latente de outra origem")
checa(cls[WF[ks]["inputs"]["model"][0]]=="ModelSamplingDiscrete",
      "modelo passa por ModelSamplingDiscrete", "sem ModelSamplingDiscrete")
checa(not [c for c in cls.values() if "IPAdapter" in c],
      "sem IP-Adapter (inpaint puro)", "IP-Adapter nao entra neste teste")
_f = WF[imc[0]]["inputs"]["mask"][0]
checa(cls[_f]=="FeatherMask", "mascara com feather", "sem feather")
checa(cls[WF[_f]["inputs"]["mask"][0]]=="GrowMask", "mascara com dilatacao", "sem GrowMask")
_aus = [c for c in set(cls.values()) if c not in OBJECT_INFO]
checa(not _aus, "todas as classes existem", f"classes ausentes: {_aus}")

print()
print("PROMPT")
import sys
sys.path.insert(0, str(SCRIPTS))
from chibi import model_registry as mr
print("  positivo (", PROMPT_SOURCE["positive"], "):", PROMPT[:70])
print("  negativo (", PROMPT_SOURCE["negative"], "):", NEGATIVE[:70])

# O prompt descreve a FUNCAO do reparo (reconstruir o figurino), nunca a
# personagem: o design especifico vem da imagem. Se a identidade vier do
# texto, a recipe deixa de servir para as outras personagens.
# So o POSITIVO e validado — no negativo, "horns" significa "evite chifres",
# que e uso legitimo.
_achados = mr.termos_especificos_no_prompt(PROMPT)
checa(not _achados, "prompt positivo generico",
      f"prompt positivo contem termos especificos de personagem: {_achados}. "
      "O design vem da imagem, nao do texto.")
checa(bool(PROMPT.strip()), "prompt positivo nao vazio", "prompt vazio")

print()
if falhas: raise SystemExit(f"BLOCKED — {len(falhas)} falha(s): {falhas}")
print("Validado.")


In [ ]:
#@title 6. Grafo resolvido (conferir antes de executar) { display-mode: "form" }
import json, copy, shutil, pathlib

COMFY_IN = pathlib.Path("/content/ComfyUI/input")
COMFY_IN.mkdir(parents=True, exist_ok=True)
shutil.copy2(SRC, COMFY_IN / SRC.name)
shutil.copy2(MSK, COMFY_IN / MSK.name)

SUBS = {
    "%%INPAINT_CKPT%%": CHECKPOINT_FILE,
    "%%SAMPLING_TYPE%%": SAMPLING_TYPE, "%%ZSNR%%": bool(ZSNR),
    "%%PROMPT%%": PROMPT, "%%NEGATIVE_PROMPT%%": NEGATIVE,
    "%%SOURCE_IMAGE%%": SRC.name, "%%OUTFIT_MASK%%": MSK.name,
    "%%MASK_CHANNEL%%": MASK_CHANNEL,
    "%%MASK_DILATION%%": int(MASK_DILATION),
    "%%MASK_FEATHER%%": int(MASK_FEATHER),
    "%%SEED%%": int(SEED), "%%STEPS%%": int(STEPS), "%%CFG%%": float(CFG_SCALE),
    "%%SAMPLER%%": SAMPLER, "%%SCHEDULER%%": SCHEDULER,
    "%%INPAINT_STRENGTH%%": float(INPAINT_STRENGTH),
    "%%OUTPUT_PREFIX%%": f"repair_{EXPERIMENT_DIR_NAME}",
}
GRAFO = copy.deepcopy(WF)
for node in GRAFO.values():
    for campo, valor in node["inputs"].items():
        if isinstance(valor, str) and valor in SUBS:
            node["inputs"][campo] = SUBS[valor]
restantes = [f"{k}.{c}" for k,n in GRAFO.items() for c,v in n["inputs"].items()
             if isinstance(v,str) and v.startswith("%%")]
assert not restantes, f"placeholders nao resolvidos: {restantes}"
print(json.dumps(GRAFO, indent=2, ensure_ascii=False))


In [ ]:
#@title 7. Executar UMA vez { display-mode: "form" }
import json, urllib.request, time, pathlib, shutil, hashlib, subprocess
import numpy as np
from PIL import Image

EXP_DIR = EVAL_ROOT / EXPERIMENT_DIR_NAME
if EXP_DIR.exists():
    raise SystemExit(f"BLOCKED — {EXP_DIR} ja existe.")
(EXP_DIR / "logs").mkdir(parents=True)

shutil.copy2(SRC, EXP_DIR / "input.png")
shutil.copy2(MSK, EXP_DIR / "mask.png")
OVERLAY.save(EXP_DIR / "overlay.png")
(EXP_DIR / "workflow.resolved.json").write_text(
    json.dumps(GRAFO, indent=2, ensure_ascii=False))

t0 = time.time()
req = urllib.request.Request("http://127.0.0.1:8188/prompt",
        data=json.dumps({"prompt": GRAFO}).encode(),
        headers={"Content-Type": "application/json"})
pid = json.load(urllib.request.urlopen(req))["prompt_id"]
print("prompt_id:", pid)
while time.time() - t0 < 1800:
    hist = json.load(urllib.request.urlopen(
        f"http://127.0.0.1:8188/history/{pid}", timeout=30))
    if pid in hist: break
    time.sleep(2)
else:
    raise SystemExit("BLOCKED — timeout")
ELAPSED = round(time.time() - t0, 1)

if hist[pid].get("status", {}).get("status_str") == "error":
    (EXP_DIR/"logs"/"error.json").write_text(json.dumps(hist[pid], indent=2))
    shutil.copy2("/content/comfy.log", EXP_DIR/"logs"/"comfy.log")
    raise SystemExit(f"BLOCKED — falhou. Log em {EXP_DIR}/logs/")

imgs = [i for o in hist[pid]["outputs"].values() for i in o.get("images", [])]
assert imgs, "nenhuma imagem"
shutil.copy2(pathlib.Path("/content/ComfyUI/output")/imgs[0]["filename"],
             EXP_DIR/"output.png")
shutil.copy2("/content/comfy.log", EXP_DIR/"logs"/"comfy.log")

def _sha(p):
    h = hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda: f.read(1<<20), b""): h.update(b)
    return h.hexdigest()

OUT = EXP_DIR/"output.png"
OUTPUT_SHA256 = _sha(OUT)
OUTPUT_PIXEL_SHA256 = hashlib.sha256(
    np.asarray(Image.open(OUT).convert("RGB")).tobytes()).hexdigest()

RECIPE = {
    **CONFIG,
    "elapsed_seconds": ELAPSED,
    "model": {
        "model_name": "Waifu-Inpaint-XL",
        "model_revision": "unknown",
        "model_sha256": CHECKPOINT_SHA256,
        "model_bytes": CHECKPOINT_BYTES,
        "model_source": "https://huggingface.co/ShinoharaHare/Waifu-Inpaint-XL",
        "license": "CreativeML Open RAIL++-M",
        "commercial_status": "pending_human_review",
        "license_note": (
            "Licenca DESTE checkpoint. Nao herda nem se mistura com a do "
            "waiIllustriousSDXL_v170, que tem termos proprios."),
        "lineage": ["KBlueLeaf/kohaku-xl-beta5",
                    "OnomaAIResearch/Illustrious-xl-early-release-v0",
                    "ShinoharaHare/WAI-NSFW-illustrious-SDXL-V14.0-V-Prediction",
                    "ShinoharaHare/Waifu-Inpaint-XL"],
        "unet_in_channels": 9,
        "prediction_type": "v_prediction",
    },
    "inputs": {
        "source_sha256": SOURCE_SHA256,
        "source_pixel_sha256": SOURCE_PIXEL_SHA256,
        "source_size": list(Image.open(SRC).size),
        "mask_sha256": MASK_SHA256,
        "protected_mask_sha256": PROTECTED_MASK_SHA256,
        "mask_area_percentage": MASK_AREA_PCT,
        "protected_overlap_pixels": PROTECTED_OVERLAP_PIXELS,
        "reference_used": None,
    },
    "workflow_sha256": WORKFLOW_SHA256,
    "artifact_sha256": OUTPUT_SHA256,
    "output_pixel_sha256": OUTPUT_PIXEL_SHA256,
    "comfyui_commit": COMFY_COMMIT,
    "custom_nodes": [],
    "custom_nodes_note": "Somente nodes de fabrica do ComfyUI.",
    "loader_node": "CheckpointLoaderSimple",
    "conditioning_node": "InpaintModelConditioning",
    "determinism_note": (
        "Mesma seed e mesmo grafo tendem a reproduzir; GPU e versoes de "
        "biblioteca podem mudar bits. Sem promessa de determinismo absoluto."),
}
(EXP_DIR/"recipe.json").write_text(json.dumps(RECIPE, indent=2, ensure_ascii=False))
print("OK em", ELAPSED, "s ->", OUT)


In [ ]:
#@title 8. Metricas de LOCALIDADE (nao de qualidade) { display-mode: "form" }
import json, sys
sys.path.insert(0, str(SCRIPTS))
from chibi.inpaint_check import comparar, diferenca_visivel, overlap_protegido
from PIL import Image
import matplotlib.pyplot as plt

METRICAS = comparar(SRC, OUT, MSK)
if PROT.exists():
    METRICAS["protected_overlap_pixels"] = overlap_protegido(MSK, PROT)
print(json.dumps(METRICAS, indent=2, ensure_ascii=False))

fig, ax = plt.subplots(1, 4, figsize=(19, 5.2))
for a, img, t in ((ax[0], Image.open(SRC), "SOURCE (run 003)"),
                  (ax[1], OVERLAY, "REGIAO EDITAVEL"),
                  (ax[2], Image.open(OUT), "OUTPUT"),
                  (ax[3], diferenca_visivel(SRC, OUT), "DIFERENCA (x8)")):
    a.imshow(img, cmap="gray" if img.mode=="L" else None)
    a.set_title(t); a.axis("off")
fig.suptitle(f"fora da mascara preservado: "
             f"{METRICAS['outside_mask_preserved_percentage']}%")
fig.tight_layout()
fig.savefig(EXP_DIR/"comparison.png", dpi=110, bbox_inches="tight")
plt.show()

(EXP_DIR/"hashes.json").write_text(json.dumps({
    "model_sha256": CHECKPOINT_SHA256,
    "source_sha256": SOURCE_SHA256,
    "source_pixel_sha256": SOURCE_PIXEL_SHA256,
    "mask_sha256": MASK_SHA256,
    "protected_mask_sha256": PROTECTED_MASK_SHA256,
    "workflow_sha256": WORKFLOW_SHA256,
    "artifact_sha256": OUTPUT_SHA256,
    "output_pixel_sha256": OUTPUT_PIXEL_SHA256,
}, indent=2))

p = METRICAS["outside_mask_preserved_percentage"]
L = ["# DESIGN REPAIR / LOCAL INPAINT — relatorio", "",
     "Linha experimental separada. O benchmark do `waiIllustriousSDXL_v170`",
     "permanece valido e intocado; este checkpoint **nao o substitui**.", "",
     "## Pergunta", "",
     "O Waifu-Inpaint-XL corrige a roupa da Run 003 sem alterar o resto?", "",
     "## Modelo", "",
     f"- nome: {RECIPE['model']['model_name']}",
     f"- sha256: `{RECIPE['model']['model_sha256']}`",
     f"- fonte: {RECIPE['model']['model_source']}",
     f"- licenca: {RECIPE['model']['license']} "
     f"({RECIPE['model']['commercial_status']})",
     f"- arquitetura: UNet {RECIPE['model']['unet_in_channels']} canais, "
     f"{RECIPE['model']['prediction_type']}",
     "- licenca propria; nao se mistura com a do WAI v17.", "",
     "## Entradas", "",
     f"- source: `{CONFIG['source_image']}` {RECIPE['inputs']['source_size']}",
     f"  - artifact `{SOURCE_SHA256}`", f"  - pixel `{SOURCE_PIXEL_SHA256}`",
     f"- mask: `{CONFIG['mask_image']}` (espaco da run 003), "
     f"{METRICAS['mask_area_percentage']}% da imagem",
     f"- overlap com regiao protegida: {RECIPE['inputs']['protected_overlap_pixels']}",
     f"- referencia declarada e NAO usada: `{CONFIG['reference_declared_not_used']}`", "",
     "## Parametros", "",
     f"- strength {CONFIG['inpaint_strength']} | steps {STEPS} | cfg {CFG_SCALE}",
     f"- {SAMPLER}/{SCHEDULER} | seed {SEED} | {SAMPLING_TYPE} | zsnr {ZSNR}",
     f"- mascara: dilation {MASK_DILATION}, feather {MASK_FEATHER}",
     f"- nodes: {RECIPE['loader_node']} + {RECIPE['conditioning_node']}",
     f"- custom nodes: {RECIPE['custom_nodes'] or 'nenhum'}",
     f"- ComfyUI: `{COMFY_COMMIT}`", "",
     "## Resultado tecnico", "",
     f"- **{p}%** dos pixels fora da mascara continuam inalterados",
     f"- fora da mascara alterado: {METRICAS['outside_mask_changed_percentage']}%",
     f"- diferenca media fora: {METRICAS['outside_mask_mean_abs_diff']}",
     f"- diferenca media dentro: {METRICAS['inside_mask_mean_abs_diff']}",
     f"- tempo: {RECIPE['elapsed_seconds']}s", "",
     "### Conclusao tecnica", ""]
if p >= 99:
    L += [f"A edicao ficou **restrita** a area mascarada ({p}% preservado)."]
elif p >= 95:
    L += [f"Edicao majoritariamente local ({p}%), com alteracao residual fora",
          "da mascara. Ha relato de terceiros de que este modelo desloca",
          "levemente a cor da imagem inteira; compativel com o observado."]
else:
    L += [f"A edicao **nao** ficou restrita a mascara: {p}% preservado.",
          "O modelo alterou area que deveria permanecer intacta. Registrar",
          "como achado — nao compensar com pos-processamento."]
L += ["", "> Estas metricas medem **localidade**, nao qualidade. Um resultado",
      "> pode ser perfeitamente local e ainda assim visualmente ruim.", "",
      "## [HUMAN REVIEW REQUIRED]", "",
      "- a roupa ficou mais fiel ao design original?",
      "- a integracao com o resto da imagem ficou boa nas bordas?",
      "- rosto, cabelo, chifres, silhueta e shading seguem preservados?", "",
      "Avaliacao estetica e humana. O agente nao decide isso."]
(EXP_DIR/"RELATORIO.md").write_text("\n".join(L) + "\n")
print()
print("\n".join(L[-9:]))


In [ ]:
#@title 9. ZIP { display-mode: "form" }
import shutil, pathlib
ZIP = "/content/waifu_inpaint_eval_results.zip"
if pathlib.Path(ZIP).exists():
    pathlib.Path(ZIP).unlink()
shutil.make_archive(ZIP[:-4], "zip", root_dir=str(EXP_DIR))
print("ZIP:", ZIP)
for p in sorted(EXP_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(EXP_DIR), f"({p.stat().st_size} bytes)")
try:
    from google.colab import files
    files.download(ZIP)
except Exception as e:
    print("download manual:", ZIP, "|", e)
print()
print("PARAR AQUI. Sem IP-Adapter, sem outros checkpoints, sem sweep.")
